## Feature Enrichment and Customer Segmentation

### Objective
Enhance the RFM dataset by incorporating additional behavioral and lifecycle features, including Coupon Ratio and Tenure, in order to improve customer segmentation and clustering performance.

- **Coupon Ratio:** measures how frequently a customer uses discounts.
- **Tenure:** represents the duration of the customer’s relationship with the company.
- **Segmentation:** classify customers based on coupon usage and tenure to capture behavioral patterns.

### Input Data
- `customer_profile_rfm.csv`: normalized RFM features.
- `Online_Sales_with_VIP.csv`: transaction-level data with VIP labeling.
- `Cleaned_CustomersData.csv`: customer information (including tenure).
- `Cleaned_E_Commerce_Dataset.csv`: additional dataset with tenure and coupon behavior.

### Methodology

1. Compute Coupon Ratio from transaction data.
2. Integrate customer tenure from multiple datasets.
3. Merge enriched features with RFM dataset.
4. Apply rule-based segmentation:
   - Coupon-based segmentation (Deal-seeker vs Loyal)
   - Tenure-based segmentation
5. Apply log transformation and standardization to prepare data for clustering.

### Output Data

The final dataset `final_integrated_data.csv` contains:
- RFM features
- Coupon Ratio
- Tenure
- Customer segments

This dataset is fully prepared for clustering and advanced customer analysis.

In [6]:
from mailcap import show

import pandas as pd
import numpy as np
from sklearn.preprocessing import StandardScaler

# load RFM data
rfm_df = pd.read_csv("../data/processed/customer_profile_rfm.csv")

# load additional datasets
sales_df = pd.read_csv("../data/processed/Online_Sales_with_VIP.csv")
customers_df = pd.read_csv("../data/cleaned_data/Cleaned_CustomersData.csv")
ecomm_df = pd.read_csv("../data/cleaned_data/Cleaned_E_Commerce_Dataset.csv")

# create coupon usage flag
sales_df["Coupon_Status"] = sales_df["Coupon_Status"].fillna("Not Used")
sales_df["Is_Coupon_Used"] = (
    sales_df["Coupon_Status"]
    .astype(str)
    .str.strip()
    .str.lower()
    .eq("used")
    .astype(int)
)

# compute coupon ratio
coupon_df = (
    sales_df
    .groupby("CustomerID")
    .agg(
        Total_Coupons_Used=("Is_Coupon_Used", "sum"),
        Total_Transactions=("Transaction_ID", "count")
    )
    .reset_index()
)

coupon_df["Coupon_Ratio"] = np.where(
    coupon_df["Total_Transactions"] > 0,
    coupon_df["Total_Coupons_Used"] / coupon_df["Total_Transactions"],
    0
)

# merge coupon ratio with customer data
dataset1 = customers_df.merge(
    coupon_df[["CustomerID", "Coupon_Ratio"]],
    on="CustomerID",
    how="left"
)

dataset1["Coupon_Ratio"] = dataset1["Coupon_Ratio"].fillna(0)

# prepare dataset 2
ecomm_df["CustomerID"] = ecomm_df["CustomerID"].astype(int)

if "Tenure" in ecomm_df.columns:
    ecomm_df = ecomm_df.rename(columns={"Tenure": "Tenure_Months"})

dataset2 = ecomm_df[["CustomerID", "Tenure_Months"]].copy()

# compute coupon ratio for dataset 2
if {"CouponUsed", "OrderCount"}.issubset(ecomm_df.columns):
    dataset2["Coupon_Ratio"] = np.where(
        ecomm_df["OrderCount"] > 0,
        ecomm_df["CouponUsed"] / ecomm_df["OrderCount"],
        0
    )
else:
    dataset2["Coupon_Ratio"] = 0

# combine extra features
extra_df = pd.concat([
    dataset1[["CustomerID", "Coupon_Ratio", "Tenure_Months"]],
    dataset2[["CustomerID", "Coupon_Ratio", "Tenure_Months"]]
], ignore_index=True)

extra_df = extra_df.groupby("CustomerID").mean().reset_index()

# merge with RFM
final_df = rfm_df.merge(extra_df, on="CustomerID", how="left")

final_df["Coupon_Ratio"] = final_df["Coupon_Ratio"].fillna(0)
final_df["Tenure_Months"] = final_df["Tenure_Months"].fillna(0)

# create segments
final_df["Coupon_Segment"] = np.where(
    final_df["Coupon_Ratio"] >= 0.5,
    "Deal-seeker",
    "Loyal"
)

final_df["Tenure_Segment"] = pd.cut(
    final_df["Tenure_Months"],
    bins=[0, 6, 12, 24, np.inf],
    labels=["New", "Short-term", "Mid-term", "Long-term"]
)

# prepare features for clustering
features = final_df[
    ["Recency", "Frequency", "Monetary", "Coupon_Ratio", "Tenure_Months"]
]

features_log = np.log1p(features)

scaler = StandardScaler()
scaled = scaler.fit_transform(features_log)

cluster_df = pd.DataFrame(
    scaled,
    columns=features.columns
)

cluster_df.insert(0, "CustomerID", final_df["CustomerID"])

display(cluster_df.head())



,CustomerID,Recency,Frequency,Monetary,Coupon_Ratio,Tenure_Months
0,12346,0.076106,-1.782571,-1.451163,3.359087,0.523628
1,12347,-0.496072,1.012911,1.486702,0.025314,-0.075799
2,12348,-0.292411,-0.266074,0.024247,0.873705,0.841183
3,12350,-1.645836,0.023983,-0.040704,0.145347,0.228137
4,12356,0.076106,0.179406,0.183935,-0.147789,0.523628


In [7]:
missing = cluster_df.isnull().sum()
missing = missing[missing > 0]

# check if cluster_df is cleaned or not.
if missing.empty:
    print("No missing values")
else:
    percent = (missing / len(cluster_df)) * 100
    result = pd.DataFrame({
        "MissingCount": missing,
        "MissingPercent": percent
    }).sort_values("MissingPercent", ascending=False)
    print(result)

No missing values


## Feature Enrichment and Customer Segmentation

Following the RFM computation, additional features are introduced to capture customer behavior beyond transaction frequency and value.

Coupon Ratio is incorporated to reflect how dependent a customer is on discounts. This helps distinguish between price-sensitive customers (deal-seekers) and those who purchase more independently (loyal customers). By adding this feature, the analysis moves beyond pure spending behavior to include promotional responsiveness.

Tenure is included to represent the length of the customer relationship. This provides lifecycle context, allowing differentiation between new and long-term customers who may exhibit different purchasing patterns.

Rather than relying solely on unsupervised clustering, simple rule-based segments (e.g., Deal-seeker vs Loyal, Tenure groups) are added to enhance interpretability. These segments act as complementary signals, helping to explain and validate clustering results.

Overall, this step transforms the dataset from purely transactional (RFM) into a more comprehensive behavioral representation, improving the quality and interpretability of downstream customer segmentation.

In [8]:
# save final dataset
cluster_df.to_csv(
    "../data/processed/final_integrated_data.csv",
    index=False
)